# 06｜自動化與排程：可重跑的分析流程（對應課程 M10）

**本 Notebook 的目標**

1. 認識兩支腳本：`scripts/clean.py`（清理）與 `scripts/weekly_report.py`（週報）
2. 用**自動化測試**守住清理邏輯：`tests/test_pipeline.py`
3. 用作業系統的排程工具，讓週報每週一早上自動產生
4. 對照講義的「人工核准關卡」：腳本只產生草稿，不自動寄出

> **AI 做法 vs 傳統做法**：課堂上用 Claude 或 ChatGPT 的排程任務，用一段提示交代要做的事；
> 傳統做法是把同樣的事寫成腳本，交給 cron（macOS／Linux）或「工作排程器」（Windows）執行。
> 兩者最好並用：先用 AI 快速試出要什麼，確定後寫成腳本長期執行。

## 1. 為什麼要從 Notebook 走到腳本？

| | Notebook | 腳本 |
| --- | --- | --- |
| 適合 | 探索、教學、說明 | 例行、排程、交接 |
| 執行方式 | 人打開、一格一格跑 | 一行指令，可排程 |
| 參數 | 改程式碼 | 命令列參數 `--week-start` |
| 出錯時 | 人看到錯誤訊息 | 回傳錯誤碼，排程工具可通知 |

本課的作法：清理邏輯寫在 `islemart_lib.py`，Notebook 和腳本**共用同一套程式**，口徑永遠一致。

In [ ]:
import subprocess
import sys
from pathlib import Path

def run(*args):
    # 用目前的 Python 執行一個指令，顯示輸出與結束代碼
    result = subprocess.run([sys.executable, *args], capture_output=True, text=True)
    print(result.stdout + result.stderr)
    print("結束代碼：", result.returncode, "（0 代表成功）")
    return result.returncode

## 2. clean.py：可稽核的清理管線

腳本會：
- 記錄每張表的列數，以及每種資料品質問題的筆數
- 執行內建檢查（訂單數不變、沒有重複、沒有負值……）
- 任何檢查不通過就以結束代碼 1 結束，**不會**輸出錯誤的檔案

In [ ]:
print(Path("scripts/clean.py").read_text(encoding="utf-8")[:1500], "...")

In [ ]:
run("scripts/clean.py", "--output", "wide_orders.csv")

## 3. 測試：改程式之前，先確認沒有改壞

`tests/test_pipeline.py` 檢查清理管線最重要的五件事。之後任何人修改清理邏輯，都要先讓測試通過。
其中一項用一筆**刻意做出來的異常資料**，確認異常規則抓得到單價多打兩個 0 的情況。

In [ ]:
run("tests/test_pipeline.py")

## 4. weekly_report.py：每週一的週報草稿

對應講義 M10 的排程提示，五件事一一寫成程式：

| 提示中的要求 | 程式中的做法 |
| --- | --- |
| 計算上週 KPI，與前四週平均比較 | `weekly_metrics()` 算 5 週，取前 4 週平均 |
| 變動超過 10% 找出影響最大的縣市或類別 | `drilldown()` 依縣市、類別拆解 |
| 依指標定義表計算 | 全部呼叫 `islemart_lib` 的指標函式 |
| 標題寫最重要的一個變化 | 取變化幅度最大的指標組成標題 |
| 不要寄出，先存草稿 | 只寫入 `drafts/`，沒有任何寄信程式 |

IsleMart 是靜態資料，所以用 `--week-start` 指定週別。先看雙 11 那一週：

In [ ]:
run("scripts/weekly_report.py", "--week-start", "2025-11-10")

In [ ]:
print(Path("drafts/weekly_report_2025-11-10.md").read_text(encoding="utf-8"))

**解讀**：雙 11 那一週訂單大增，延遲率也跟著上升，腳本自動找出延遲惡化最多的縣市與類別。

再看資料最後一個完整週。注意報告最後的提醒：**最近一週的訂單可能還沒全部送達**，
已送達的多半是送得快的，延遲率會被**低估**。這就是 M3 的「時效性」陷阱，腳本必須把它寫在報告上。

In [ ]:
run("scripts/weekly_report.py", "--week-start", "2025-12-15")

腳本也會擋下不合理的輸入，例如不是週一、或資料還不完整的週：

In [ ]:
run("scripts/weekly_report.py", "--week-start", "2025-12-29")

## 5. 排程：每週一 08:30 自動執行

以下指令請在**終端機**執行（不是在 Notebook 裡）。路徑請換成你電腦上的實際位置。

### macOS／Linux：cron

```bash
crontab -e
```

加入一行（每週一 08:30 執行，輸出記錄到 logs/weekly.log）：

```bash
30 8 * * 1  cd /你的路徑/Python實作範例 && /usr/bin/python3 scripts/weekly_report.py >> logs/weekly.log 2>&1
```

> macOS 第一次使用 cron 時，可能需要在「系統設定 → 隱私權與安全性 → 完整磁碟取用權限」允許 cron。

### Windows：工作排程器

在「命令提示字元」執行：

```bat
schtasks /Create /SC WEEKLY /D MON /ST 08:30 /TN "IsleMart週報" ^
  /TR "C:\Python312\python.exe C:\你的路徑\Python實作範例\scripts\weekly_report.py"
```

也可以打開「工作排程器」圖形介面，依序設定觸發程序（每週一 08:30）與動作（執行 python.exe 加上腳本路徑）。

## 6. 人工核准關卡

對照講義 M10 的三欄分類，這兩支腳本刻意這樣設計：

| 可以自動做 | 先給人確認 | 不交給程式 |
| --- | --- | --- |
| 讀資料、清理、計算 KPI | 寄出週報（目前只產生草稿） | 改寫原始資料（腳本只讀不寫） |
| 產生圖表與草稿 | 發到群組或共用資料夾 | 對外發布 |
| 檢查不通過就停下來 | 修改門檻或指標定義 | 涉及個資的匯出 |

**最小權限**：排程用的資料帳號只開唯讀；腳本只寫入自己的輸出資料夾。

## 小結：從 AI 排程任務到傳統排程

| 課堂上的 AI 做法 | 本 Notebook 的傳統做法 |
| --- | --- |
| 用一段提示建立排程任務 | `weekly_report.py` ＋ cron／工作排程器 |
| 請 AI 先存草稿、不要寄出 | 腳本只寫 `drafts/`，沒有寄信功能 |
| 請 AI 自我檢查 | `run_checks()` 與 `tests/`，不通過就停止 |
| Claude Code 把步驟寫成腳本 | 就是這兩支腳本：可稽核、可重跑、可交接 |

**練習**：
1. 在 `weekly_report.py` 加一個指標「平均客單價」，並在 `tests/` 加一項對應的測試。
2. 把提醒門檻改成 5%，看看雙 11 那週會多出哪些提醒。